# Model Training (Decision Tree Algorithm)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import joblib

In [2]:
df = pd.read_csv("insurance_fraud_preprocessed_without_scaled.csv")

print("Dataset Shape:", df.shape)

Dataset Shape: (11988, 37)


In [3]:
df.head()

,age_of_driver,gender,marital_status,safety_rating,annual_income,high_education,address_change,property_status,zip_code,claim_date,...,claim_day_of_week_Friday,claim_day_of_week_Monday,claim_day_of_week_Saturday,claim_day_of_week_Sunday,claim_day_of_week_Thursday,claim_day_of_week_Tuesday,claim_day_of_week_Wednesday,vehicle_category_Compact,vehicle_category_Large,vehicle_category_Medium
0,39,1,1,73,58612.8,1,0,0,50048,2023-08-12,...,0,0,1,0,0,0,0,0,1,0
1,33,1,1,72,35936.0,0,1,0,50006,2024-10-18,...,1,0,0,0,0,0,0,0,0,1
2,31,1,1,76,84940.8,1,1,0,15021,2023-12-23,...,0,0,1,0,0,0,0,0,0,1
3,53,0,1,93,73526.4,0,1,0,85027,2024-02-25,...,0,0,0,1,0,0,0,0,0,1
4,41,1,1,87,59403.2,1,0,0,80046,2024-09-04,...,0,0,0,0,0,0,1,0,0,1


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11988 entries, 0 to 11987
Data columns (total 37 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   age_of_driver                11988 non-null  int64  
 1   gender                       11988 non-null  int64  
 2   marital_status               11988 non-null  int64  
 3   safety_rating                11988 non-null  int64  
 4   annual_income                11988 non-null  float64
 5   high_education               11988 non-null  int64  
 6   address_change               11988 non-null  int64  
 7   property_status              11988 non-null  int64  
 8   zip_code                     11988 non-null  int64  
 9   claim_date                   11988 non-null  object 
 10  past_num_of_claims           11988 non-null  int64  
 11  witness_present              11988 non-null  int64  
 12  liab_prct                    11988 non-null  int64  
 13  police_report   

In [6]:
print(df.columns.tolist())

['age_of_driver', 'gender', 'marital_status', 'safety_rating', 'annual_income', 'high_education', 'address_change', 'property_status', 'zip_code', 'claim_date', 'past_num_of_claims', 'witness_present', 'liab_prct', 'police_report', 'age_of_vehicle', 'vehicle_price', 'total_claim', 'injury_claim', 'policy deductible', 'annual premium', 'days open', 'form defects', 'fraud reported', 'claim_day', 'claim_month', 'claim_year', 'is_weekend', 'claim_day_of_week_Friday', 'claim_day_of_week_Monday', 'claim_day_of_week_Saturday', 'claim_day_of_week_Sunday', 'claim_day_of_week_Thursday', 'claim_day_of_week_Tuesday', 'claim_day_of_week_Wednesday', 'vehicle_category_Compact', 'vehicle_category_Large', 'vehicle_category_Medium']


In [7]:
missing_values = df.isnull().sum()

print(missing_values[missing_values > 0])

Series([], dtype: int64)


In [8]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [9]:
print(df["fraud reported"].value_counts())

fraud reported
0    9033
1    2947
2       8
Name: count, dtype: int64


In [10]:
print(df["fraud reported"].value_counts(normalize=True))

fraud reported
0    0.753504
1    0.245829
2    0.000667
Name: proportion, dtype: float64


In [11]:
print(df[df["fraud reported"] == 2])

       age_of_driver  gender  marital_status  safety_rating  annual_income  \
1286              32       0               0             82        55448.0   
3205              48       0               1             84        61915.2   
4729              51       0               1             87        62924.8   
5479              24       0               1             58        50859.2   
8258              53       1               1             74        63508.8   
9310              54       1               0             88        63840.0   
9855              37       0               1             83        57814.4   
10375             56       1               1             77        64401.6   

       high_education  address_change  property_status  zip_code  claim_date  \
1286                1               0                1     85077  2024-09-11   
3205                1               0                1     85062  2024-11-27   
4729                1               1                0   

In [12]:
df = df[df["fraud reported"] != 2].copy()

In [13]:
print(df["fraud reported"].value_counts())

fraud reported
0    9033
1    2947
Name: count, dtype: int64


In [14]:
df = df.drop(columns=["claim_date"])

In [15]:
print(df.shape)

(11980, 36)


In [16]:
X = df.drop(columns=["fraud reported"])
y = df["fraud reported"]

In [17]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (11980, 35)
y shape: (11980,)


In [19]:
print("Numerical columns:")
print(X.select_dtypes(include=np.number).columns.tolist())

Numerical columns:
['age_of_driver', 'gender', 'marital_status', 'safety_rating', 'annual_income', 'high_education', 'address_change', 'property_status', 'zip_code', 'past_num_of_claims', 'witness_present', 'liab_prct', 'police_report', 'age_of_vehicle', 'vehicle_price', 'total_claim', 'injury_claim', 'policy deductible', 'annual premium', 'days open', 'form defects', 'claim_day', 'claim_month', 'claim_year', 'is_weekend', 'claim_day_of_week_Friday', 'claim_day_of_week_Monday', 'claim_day_of_week_Saturday', 'claim_day_of_week_Sunday', 'claim_day_of_week_Thursday', 'claim_day_of_week_Tuesday', 'claim_day_of_week_Wednesday', 'vehicle_category_Compact', 'vehicle_category_Large', 'vehicle_category_Medium']


In [20]:
scale_columns = [
    "age_of_driver",
    "safety_rating",
    "annual_income",
    "zip_code",
    "past_num_of_claims",
    "liab_prct",
    "age_of_vehicle",
    "vehicle_price",
    "total_claim",
    "injury_claim",
    "policy deductible",
    "annual premium",
    "days open",
    "form defects",
    "claim_day",
    "claim_month",
    "claim_year"
]

In [21]:
missing_scale_columns = [
    col for col in scale_columns
    if col not in X.columns
]

print("Missing scaling columns:", missing_scale_columns)

Missing scaling columns: []


In [22]:
numeric_columns = X.select_dtypes(
    include=np.number
).columns.tolist()

print("All numerical columns:")
print(numeric_columns)

print("\nNumerical columns being scaled:")
print(scale_columns)

print("\nNumerical columns NOT being scaled:")
print([
    col for col in numeric_columns
    if col not in scale_columns
])

All numerical columns:
['age_of_driver', 'gender', 'marital_status', 'safety_rating', 'annual_income', 'high_education', 'address_change', 'property_status', 'zip_code', 'past_num_of_claims', 'witness_present', 'liab_prct', 'police_report', 'age_of_vehicle', 'vehicle_price', 'total_claim', 'injury_claim', 'policy deductible', 'annual premium', 'days open', 'form defects', 'claim_day', 'claim_month', 'claim_year', 'is_weekend', 'claim_day_of_week_Friday', 'claim_day_of_week_Monday', 'claim_day_of_week_Saturday', 'claim_day_of_week_Sunday', 'claim_day_of_week_Thursday', 'claim_day_of_week_Tuesday', 'claim_day_of_week_Wednesday', 'vehicle_category_Compact', 'vehicle_category_Large', 'vehicle_category_Medium']

Numerical columns being scaled:
['age_of_driver', 'safety_rating', 'annual_income', 'zip_code', 'past_num_of_claims', 'liab_prct', 'age_of_vehicle', 'vehicle_price', 'total_claim', 'injury_claim', 'policy deductible', 'annual premium', 'days open', 'form defects', 'claim_day', 'clai

In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (9584, 35)
X_test : (2396, 35)
y_train: (9584,)
y_test : (2396,)


In [24]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

In [25]:
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[scale_columns] = scaler.fit_transform(
    X_train[scale_columns]
)

X_test_scaled[scale_columns] = scaler.transform(
    X_test[scale_columns]
)

In [27]:
print("Before scaling:")
X_train[scale_columns].head()

Before scaling:


,age_of_driver,safety_rating,annual_income,zip_code,past_num_of_claims,liab_prct,age_of_vehicle,vehicle_price,total_claim,injury_claim,policy deductible,annual premium,days open,form defects,claim_day,claim_month,claim_year
7856,40,66,59032.00,20117,3,37,2,38545.771440,35318.68172,4789.784445,1000,1503.21,10.115659,4,26.0,5.0,2024.0
9252,81,91,98425.60,50011,0,98,1,9474.688093,36892.86295,8001.602802,500,1078.65,10.672839,4,24.0,3.0,2023.0
10508,49,66,87151.68,80030,4,25,5,20031.585310,35474.06950,6529.398865,500,1232.57,8.730594,5,9.0,1.0,2024.0
8632,66,74,67049.60,80042,1,95,6,20959.941300,15334.11346,3883.977109,500,1082.36,11.717235,3,19.0,10.0,2024.0
5189,29,50,53870.40,85077,3,19,3,15479.003240,19222.56635,8569.962463,500,1082.36,4.632782,4,23.0,11.0,2024.0


In [28]:
print("After scaling:")
X_train_scaled[scale_columns].head()

After scaling:


,age_of_driver,safety_rating,annual_income,zip_code,past_num_of_claims,liab_prct,age_of_vehicle,vehicle_price,total_claim,injury_claim,policy deductible,annual premium,days open,form defects,claim_day,claim_month,claim_year
7856,-0.284313,-0.492272,-0.265362,-1.013978,2.616056,-0.383363,-1.335421,1.293884,1.016920,-0.106981,-0.016220,1.007952,0.546731,0.012863,1.081806,-0.441337,1.018748
9252,3.053978,1.123392,1.979881,0.011758,-0.514457,1.427954,-1.785539,-1.110908,1.145515,1.293330,-0.904521,-0.820101,0.778576,0.012863,0.859764,-1.022549,-0.981597
10508,0.448482,-0.492272,1.337323,1.041784,3.659560,-0.739688,0.014935,-0.237630,1.029613,0.651468,-0.904521,-0.157359,-0.029598,0.516031,-0.805551,-1.603762,1.018748
8632,1.832652,0.024741,0.191602,1.042195,0.529047,1.338873,0.465054,-0.160835,-0.615632,-0.501901,-0.904521,-0.804127,1.213152,-0.490306,0.304659,1.011694,1.018748
5189,-1.179953,-1.526297,-0.559548,1.214959,2.616056,-0.917851,-0.885302,-0.614225,-0.297982,1.541127,-0.904521,-0.804127,-1.734710,0.012863,0.748743,1.302300,1.018748


In [31]:
from sklearn.tree import DecisionTreeClassifier

dt_baseline = DecisionTreeClassifier(
    criterion="gini",
    random_state=42
)

dt_baseline.fit(
    X_train_scaled,
    y_train
)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [32]:
y_train_pred = dt_baseline.predict(X_train_scaled)
y_test_pred = dt_baseline.predict(X_test_scaled)

In [33]:
y_train_pred

array([0, 0, 0, ..., 1, 0, 0], shape=(9584,))

In [34]:
y_test_pred

array([0, 0, 0, ..., 0, 0, 1], shape=(2396,))

In [35]:
from sklearn.metrics import accuracy_score

train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print("Training Accuracy:", train_accuracy)
print("Testing Accuracy :", test_accuracy)

Training Accuracy: 1.0
Testing Accuracy : 0.6636060100166945


In [36]:
from sklearn.metrics import precision_score, recall_score, f1_score

precision = precision_score(y_test, y_test_pred)
recall = recall_score(y_test, y_test_pred)
f1 = f1_score(y_test, y_test_pred)

print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

Precision: 0.32242225859247137
Recall   : 0.33446519524617996
F1 Score : 0.3283333333333333


In [37]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_test_pred,
        target_names=["Not Fraud", "Fraud"]
    )
)

              precision    recall  f1-score   support

   Not Fraud       0.78      0.77      0.78      1807
       Fraud       0.32      0.33      0.33       589

    accuracy                           0.66      2396
   macro avg       0.55      0.55      0.55      2396
weighted avg       0.67      0.66      0.67      2396



In [40]:
# from sklearn.metrics import confusion_matrix

# cm = confusion_matrix(y_test, y_test_pred)

# print(cm)

In [39]:
print("Tree Depth:", dt_baseline.get_depth())
print("Number of Leaves:", dt_baseline.get_n_leaves())

Tree Depth: 36
Number of Leaves: 1500


In [41]:
# With max_depth = 3
dt_depth3 = DecisionTreeClassifier(
    criterion="gini",
    max_depth=3,
    random_state=42
)

dt_depth3.fit(X_train_scaled, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,3
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [42]:
y_train_pred_3 = dt_depth3.predict(X_train_scaled)
y_test_pred_3 = dt_depth3.predict(X_test_scaled)

In [43]:
print("Training Accuracy:", accuracy_score(y_train, y_train_pred_3))
print("Testing Accuracy :", accuracy_score(y_test, y_test_pred_3))
print("Precision         :", precision_score(y_test, y_test_pred_3))
print("Recall            :", recall_score(y_test, y_test_pred_3))
print("F1 Score          :", f1_score(y_test, y_test_pred_3))

Training Accuracy: 0.7824499165275459
Testing Accuracy : 0.7750417362270451
Precision         : 0.8787878787878788
Recall            : 0.09847198641765705
F1 Score          : 0.17709923664122137


In [44]:
print("Tree Depth:", dt_depth3.get_depth())
print("Number of Leaves:", dt_depth3.get_n_leaves())

Tree Depth: 3
Number of Leaves: 8


In [45]:
# With max_depth = 5
dt_depth5 = DecisionTreeClassifier(
    criterion="gini",
    max_depth=5,
    random_state=42
)

dt_depth5.fit(X_train_scaled, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,5
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [46]:
y_train_pred_5 = dt_depth5.predict(X_train_scaled)
y_test_pred_5 = dt_depth5.predict(X_test_scaled)

In [47]:
print("Tree Depth:", dt_depth5.get_depth())
print("Number of Leaves:", dt_depth5.get_n_leaves())

Tree Depth: 5
Number of Leaves: 25


In [48]:
print("Training Accuracy:", accuracy_score(y_train, y_train_pred_5))
print("Testing Accuracy :", accuracy_score(y_test, y_test_pred_5))
print("Precision         :", precision_score(y_test, y_test_pred_5))
print("Recall            :", recall_score(y_test, y_test_pred_5))
print("F1 Score          :", f1_score(y_test, y_test_pred_5))

Training Accuracy: 0.7838063439065108
Testing Accuracy : 0.7767111853088481
Precision         : 0.921875
Recall            : 0.100169779286927
F1 Score          : 0.1807044410413476


In [49]:
# With max_depth = 10
dt_depth10 = DecisionTreeClassifier(
    criterion="gini",
    max_depth=10,
    random_state=42
)

dt_depth10.fit(X_train_scaled, y_train)

y_train_pred_10 = dt_depth10.predict(X_train_scaled)
y_test_pred_10 = dt_depth10.predict(X_test_scaled)

print("Training Accuracy:", accuracy_score(y_train, y_train_pred_10))
print("Testing Accuracy :", accuracy_score(y_test, y_test_pred_10))
print("Precision         :", precision_score(y_test, y_test_pred_10))
print("Recall            :", recall_score(y_test, y_test_pred_10))
print("F1 Score          :", f1_score(y_test, y_test_pred_10))

print("Tree Depth:", dt_depth10.get_depth())
print("Number of Leaves:", dt_depth10.get_n_leaves())

Training Accuracy: 0.7974749582637729
Testing Accuracy : 0.7658597662771286
Precision         : 0.6346153846153846
Recall            : 0.11205432937181664
F1 Score          : 0.19047619047619047
Tree Depth: 10
Number of Leaves: 132


In [53]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr_model.fit(X_train_scaled, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [55]:
y_train_pred_lr = lr_model.predict(X_train_scaled)
y_test_pred_lr = lr_model.predict(X_test_scaled)

In [57]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("LOGISTIC REGRESSION")

print("Training Accuracy:", accuracy_score(y_train, y_train_pred_lr))
print("Testing Accuracy :", accuracy_score(y_test, y_test_pred_lr))
print("Precision         :", precision_score(y_test, y_test_pred_lr))
print("Recall            :", recall_score(y_test, y_test_pred_lr))
print("F1 Score          :", f1_score(y_test, y_test_pred_lr))

LOGISTIC REGRESSION
Training Accuracy: 0.7779632721202003
Testing Accuracy : 0.7708681135225376
Precision         : 0.7564102564102564
Recall            : 0.100169779286927
F1 Score          : 0.17691154422788605


In [58]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_test_pred_lr,
        target_names=["Not Fraud", "Fraud"]
    )
)

              precision    recall  f1-score   support

   Not Fraud       0.77      0.99      0.87      1807
       Fraud       0.76      0.10      0.18       589

    accuracy                           0.77      2396
   macro avg       0.76      0.54      0.52      2396
weighted avg       0.77      0.77      0.70      2396



In [64]:
from sklearn.metrics import confusion_matrix
cm_lr = confusion_matrix(y_test, y_test_pred_lr)
print(cm_lr)

[[1788   19]
 [ 530   59]]


In [59]:
from sklearn.neighbors import KNeighborsClassifier
knn_model = KNeighborsClassifier(
    n_neighbors=5
)
knn_model.fit(
    X_train_scaled,
    y_train
)

,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [60]:
y_train_pred_knn = knn_model.predict(X_train_scaled)
y_test_pred_knn = knn_model.predict(X_test_scaled)

In [61]:
print("K-NEAREST NEIGHBORS")

print("Training Accuracy:", accuracy_score(y_train, y_train_pred_knn))
print("Testing Accuracy :", accuracy_score(y_test, y_test_pred_knn))
print("Precision         :", precision_score(y_test, y_test_pred_knn))
print("Recall            :", recall_score(y_test, y_test_pred_knn))
print("F1 Score          :", f1_score(y_test, y_test_pred_knn))

K-NEAREST NEIGHBORS
Training Accuracy: 0.7903797996661102
Testing Accuracy : 0.7128547579298832
Precision         : 0.2946058091286307
Recall            : 0.12054329371816638
F1 Score          : 0.1710843373493976


In [62]:
print(
    classification_report(
        y_test,
        y_test_pred_knn,
        target_names=["Not Fraud", "Fraud"]
    )
)

              precision    recall  f1-score   support

   Not Fraud       0.76      0.91      0.83      1807
       Fraud       0.29      0.12      0.17       589

    accuracy                           0.71      2396
   macro avg       0.53      0.51      0.50      2396
weighted avg       0.65      0.71      0.67      2396



In [63]:
cm_knn = confusion_matrix(y_test, y_test_pred_knn)
print(cm_knn)

[[1637  170]
 [ 518   71]]


In [65]:
from sklearn.ensemble import RandomForestClassifier
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)
rf_model.fit(
    X_train_scaled,
    y_train
)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [66]:
y_train_pred_rf = rf_model.predict(X_train_scaled)
y_test_pred_rf = rf_model.predict(X_test_scaled)

In [67]:
print("RANDOM FOREST")

print("Training Accuracy:", accuracy_score(y_train, y_train_pred_rf))
print("Testing Accuracy :", accuracy_score(y_test, y_test_pred_rf))
print("Precision         :", precision_score(y_test, y_test_pred_rf))
print("Recall            :", recall_score(y_test, y_test_pred_rf))
print("F1 Score          :", f1_score(y_test, y_test_pred_rf))

RANDOM FOREST
Training Accuracy: 1.0
Testing Accuracy : 0.7758764607679466
Precision         : 0.8513513513513513
Recall            : 0.10696095076400679
F1 Score          : 0.19004524886877827


In [68]:
print(
    classification_report(
        y_test,
        y_test_pred_rf,
        target_names=["Not Fraud", "Fraud"]
    )
)

              precision    recall  f1-score   support

   Not Fraud       0.77      0.99      0.87      1807
       Fraud       0.85      0.11      0.19       589

    accuracy                           0.78      2396
   macro avg       0.81      0.55      0.53      2396
weighted avg       0.79      0.78      0.70      2396



In [69]:
cm_rf = confusion_matrix(y_test, y_test_pred_rf)
print(cm_rf)

[[1796   11]
 [ 526   63]]


In [70]:
comparison_df = pd.DataFrame({
    "Model": [
        "Decision Tree",
        "Logistic Regression",
        "KNN",
        "Random Forest"
    ],

    "Training Accuracy": [
        accuracy_score(y_train, y_train_pred),
        accuracy_score(y_train, y_train_pred_lr),
        accuracy_score(y_train, y_train_pred_knn),
        accuracy_score(y_train, y_train_pred_rf)
    ],

    "Testing Accuracy": [
        accuracy_score(y_test, y_test_pred),
        accuracy_score(y_test, y_test_pred_lr),
        accuracy_score(y_test, y_test_pred_knn),
        accuracy_score(y_test, y_test_pred_rf)
    ],

    "Precision": [
        precision_score(y_test, y_test_pred),
        precision_score(y_test, y_test_pred_lr),
        precision_score(y_test, y_test_pred_knn),
        precision_score(y_test, y_test_pred_rf)
    ],

    "Recall": [
        recall_score(y_test, y_test_pred),
        recall_score(y_test, y_test_pred_lr),
        recall_score(y_test, y_test_pred_knn),
        recall_score(y_test, y_test_pred_rf)
    ],

    "F1 Score": [
        f1_score(y_test, y_test_pred),
        f1_score(y_test, y_test_pred_lr),
        f1_score(y_test, y_test_pred_knn),
        f1_score(y_test, y_test_pred_rf)
    ]
})

comparison_df

,Model,Training Accuracy,Testing Accuracy,Precision,Recall,F1 Score
0,Decision Tree,1.000000,0.663606,0.322422,0.334465,0.328333
1,Logistic Regression,0.777963,0.770868,0.756410,0.100170,0.176912
2,KNN,0.790380,0.712855,0.294606,0.120543,0.171084
3,Random Forest,1.000000,0.775876,0.851351,0.106961,0.190045
